### Distance-3 rotated surface code
This notebook builds a memory experiment for the distance-3 rotated surface code on qudits of dimension $d$: a few rounds of stabilizer measurements with depolarizing noise on the data, then a readout of the data. Detectors compare each check with its value in the previous round, and the logical observable is a logical $Z$ operator read from the data. The circuit is then sampled two ways, with the Pauli frame simulator and with a detector error model from `sdim.dem`.

In [ ]:
import numpy as np
from sdim import Circuit, Program
from sdim.dem import DetectorErrorModel

### Layout
Nine data qudits sit on a $3 \times 3$ grid, numbered row by row. Each check has its own ancilla: four measure $Z$-type stabilizers and four measure $X$-type stabilizers (two of each kind have weight 4, two have weight 2 on the boundary).

For qubits, an $X$-type and a $Z$-type check commute because they overlap on an even number of qubits. For $d > 2$, $X^{a_1} \otimes X^{a_2} \otimes \cdots$ and $Z^{b_1} \otimes Z^{b_2} \otimes \cdots$ commute when $\sum_j a_j b_j = 0 \bmod d$. Here every $X$-type check uses power 1, and the $Z$-type checks use powers $\pm 1$ chosen so that each overlap contributes $1 - 1 = 0$.

In [ ]:
d = 3        # qudit dimension; sdim.dem needs a prime, the frame sampler takes any d
rounds = 3   # rounds of stabilizer measurements
p = 0.01     # depolarizing probability on each data qudit before each round

data = list(range(9))  # 0 1 2 / 3 4 5 / 6 7 8
z_ancillas = [9, 10, 11, 12]
x_ancillas = [13, 14, 15, 16]

# Z-type checks as {data qudit: power of Z}
z_checks = [{1: 1, 2: -1, 4: -1, 5: 1}, {3: -1, 4: 1, 6: 1, 7: -1}, {0: 1, 3: -1}, {5: 1, 8: -1}]
# X-type checks, with power 1 on each of their data qudits
x_checks = [[0, 1, 3, 4], [4, 5, 7, 8], [1, 2], [6, 7]]
# Logical Z^1 Z^-1 Z^1 along the top row, which commutes with every check
logical_z = {0: 1, 1: -1, 2: 1}

### The circuit
A $Z$-type ancilla starts in $|0\rangle$ and collects $\sum_j b_j x_j$ through `CNOT` (power 1) and `CNOT_INV` (power $-1$). An $X$-type ancilla is put in superposition with `H`, controls a `CNOT` onto each of its data qudits, and is rotated back with `H_INV`, so that measuring it reads out the $X$-type check.

The data starts in $|0 \cdots 0\rangle$, so the $Z$-type checks are deterministic from the first round, while the $X$-type checks start random and only get detectors from the second round on. Detectors refer to measurement records by absolute index, `rec[k]` being the k-th measurement of the circuit.

In [ ]:
def signed_sum(powers):
    """Detector expression for the sum of power * rec[index], with powers +1 and -1."""
    terms = [("- " if power == -1 else "+ ") + f"rec[{index}]" for index, power in powers.items()]
    return " ".join(terms).removeprefix("+ ")

circuit = Circuit(17, d)
measured = 0      # number of measurements so far
previous = None   # records of the previous round: the 4 Z-type checks, then the 4 X-type checks
for r in range(rounds):
    circuit.add_gate("N1", data, prob=p)
    circuit.add_gate("RESET", z_ancillas + x_ancillas)
    for ancilla, check in zip(z_ancillas, z_checks):
        for q, power in check.items():
            circuit.add_gate("CNOT" if power == 1 else "CNOT_INV", q, ancilla)
    circuit.add_gate("H", x_ancillas)
    for ancilla, check in zip(x_ancillas, x_checks):
        circuit.add_gate("CNOT", ancilla, check)
    circuit.add_gate("H_INV", x_ancillas)
    circuit.add_gate("M", z_ancillas + x_ancillas)
    current = list(range(measured, measured + 8))
    measured += 8
    for k in range(8):
        if previous is not None:
            circuit.add_gate("DETECTOR", expr=f"rec[{current[k]}] - rec[{previous[k]}]")
        elif k < 4:
            circuit.add_gate("DETECTOR", expr=f"rec[{current[k]}]")
    previous = current

# Measure the data, rebuild each Z-type check from it, and read out the logical observable.
circuit.add_gate("M", data)
final = {q: measured + q for q in data}
for k, check in enumerate(z_checks):
    expr = signed_sum({final[q]: power for q, power in check.items()})
    circuit.add_gate("DETECTOR", expr=f"{expr} - rec[{previous[k]}]")
circuit.add_gate("LOGICAL_OBSERVABLE", expr=signed_sum({final[q]: power for q, power in logical_z.items()}))

### Detector error model
`DetectorErrorModel.from_circuit` checks that every detector and the observable are deterministic without noise, then turns the noise gates into independent error mechanisms, one per line. A line `ERROR(pi) g_1 | g_2` fires with probability `pi`. When it fires, it adds $a_1 g_1 + a_2 g_2$ to the detectors (`Dk`) and the observable (`L0`), mod $d$, where each $a_j$ is uniformly random mod $d$, zero included. So for depolarizing noise of strength $p$, `pi` is $p \, d^2 / (d^2 - 1)$, a little more than $p$: one of the $d^2$ equally likely Paulis is the identity. Mechanisms with a single generator (here those of the first round's noise, whose $Z$ errors no detector sees) are merged into one line when their generators are multiples of each other, and the comment names the gates each line came from. Mechanisms with more generators are kept as they are.

In [ ]:
dem = DetectorErrorModel.from_circuit(circuit)
print("\n".join(str(dem).splitlines()[:10]))

### Sampling
Both samplers should give the same statistics. `dem.sample` returns arrays indexed as (shot, detector). The frame sampler returns its detector arrays the other way round, (detector, shot), for the `shots - 1` shots after its noiseless reference shot. A decoder would use the detectors to correct the observable; here we only count how often it is nonzero without correction.

In [ ]:
shots = 20_000
detectors, observables = dem.sample(shots, seed=1)
_, (frame_detectors, frame_observables) = Program(circuit).simulate(shots=shots + 1, raw_detector_output=True)

print(f"detection events per shot: DEM {np.mean(np.sum(detectors != 0, axis=1)):.3f}, "
      f"frame sampler {np.mean(np.sum(frame_detectors != 0, axis=0)):.3f}")
print(f"shots with a nonzero observable: DEM {np.mean(observables[:, 0] != 0):.4f}, "
      f"frame sampler {np.mean(frame_observables[0] != 0):.4f}")